# Imports

In [1]:
!pip install transformers torch datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.7 MB/s eta 0:00:00


# Training Setup

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Target Model

name_t = "Qwen/Qwen2.5-7B-Instruct"

tokenizer_t = AutoTokenizer.from_pretrained(name_t)
model_t = AutoModelForCausalLM.from_pretrained(
    name_t,
    torch_dtype=torch.float16,
    device_map="auto"
)

model_t.eval()


# Draft Model

name_d = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_d = AutoTokenizer.from_pretrained(name_d)
model_d = AutoModelForCausalLM.from_pretrained(
    name_d,
    torch_dtype=torch.float16,
    device_map="auto"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# Dataset Pre-processing

In [3]:
from datasets import load_dataset
from torch.utils.data import DataLoader

acds = load_dataset("sahil2801/CodeAlpaca-20k", split="train")

def format_sample(sample):
    user_content = sample["instruction"]
    if sample["input"].strip():
        user_content += "\n\n" + sample["input"]

    msgs = [
        { "role": "system",    "content": "You are a coding assistant" },
        { "role": "user",      "content": user_content },
        { "role": "assistant", "content": sample["output"] }
    ]

    formatted_sample = tokenizer_d.apply_chat_template(  # ✅ use draft tokenizer
        msgs,
        tokenize=False,
        add_generation_prompt=False
    )

    return { "text": formatted_sample }

preprocessed_dataset = acds.map(format_sample, remove_columns=acds.column_names)


def tokenize_sample(sample):
    tokens = tokenizer_d(                        # ✅ use draft tokenizer
        sample["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    labels = tokens["input_ids"].copy()

    # ✅ Mask padding using the actual pad token id
    pad_id = tokenizer_d.pad_token_id
    # print(f"pad_token_id = {pad_id}")           # verify this once
    labels[labels == pad_id] = -100

    tokens["labels"] = labels
    return tokens


tokenized_dataset = preprocessed_dataset.map(
    tokenize_sample,
    batched=False,                               # ✅ batched=False for safety
    remove_columns=["text"]
)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

dataloader = DataLoader(tokenized_dataset, batch_size=2, shuffle=True)


README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

Map:   0%|          | 0/20022 [00:00<?, ? examples/s]

Map:   0%|          | 0/20022 [00:00<?, ? examples/s]

# Knowledge Distillation

In [46]:
import torch.nn.functional as F
from torch.optim import AdamW

from google.colab import drive

import os
import shutil

drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/distillation/checkpoints/"
SAVE_DIR = "/content/drive/MyDrive/distillation/weights/"
SAVE_EVERY_N = 500
CHECKPOINT_WINDOW_LEN = 3

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

optimizer = AdamW(model_d.parameters(), lr=2e-7)

TEMP = 2.0
ALPHA = 0.9
EPOCHS = 3
K = 50


if not torch.cuda.is_available():
  print("ERROR: Need GPU")
  exit(1)

device="cuda"

model_t.eval()
model_d.train()

for epoch in range(EPOCHS):

  model_d.train()

  total_loss = 0
  total_dist_loss = 0
  total_sample_loss = 0

  for step, batch in enumerate(dataloader):

    input_ids = batch["input_ids"].to(device) # Move batches ids to gpu
    attention_mask = batch["attention_mask"].to(device) # Move padding masks to gpu
    print(attention_mask.shape)
    labels = batch["labels"].to(device)

    with torch.no_grad():
      output_t = model_t(
          input_ids=input_ids,
          attention_mask=attention_mask
      )

      target_logits = output_t.logits.float()

    output_d = model_d(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    draft_logits = output_d.logits.float()


    sample_loss = output_d.loss

    # ✅ Guard against exploding loss before backward
    if sample_loss.isnan() or sample_loss.isinf():
        print(f"⚠️ Skipping step {step} — loss={sample_loss.item()}")
        optimizer.zero_grad()
        continue

    _, topk_idx = torch.topk(target_logits, k=K, dim=-1)
    topk_logits_t = target_logits.gather(-1, topk_idx)
    topk_logits_d = draft_logits.gather(-1, topk_idx)


    log_prob_dist_t = F.log_softmax(topk_logits_t / TEMP, dim=-1)
    log_prob_dist_d = F.log_softmax(topk_logits_d / TEMP, dim=-1)

    # ✅ ADD THESE
    print(f"log_prob_t NaN: {log_prob_dist_t.isnan().any().item()}")
    print(f"log_prob_d NaN: {log_prob_dist_d.isnan().any().item()}")
    print(f"log_prob_t min: {log_prob_dist_t.min().item():.4f}")
    print(f"log_prob_d min: {log_prob_dist_d.min().item():.4f}")

    distill_loss = F.kl_div(
        log_prob_dist_d,
        log_prob_dist_t,
        reduction="batchmean",
        log_target=True
    ) * (TEMP ** 2)

    # ✅ Normalize by sequence length to account for all 512 positions
    #SEQ_LEN = input_ids.size(1)
    #distill_loss = distill_loss / SEQ_LEN

      # ✅ ADD THESE BEFORE COMBINING
    print(f"distill_loss: {distill_loss.item():.4f}")
    print(f"sample_loss:  {sample_loss.item():.4f}")

    loss = ALPHA * distill_loss + (1 - ALPHA) * sample_loss

    # ✅ Guard against exploding loss before backward
    if loss.isnan() or loss.isinf() or loss.item() > 100:
        print(f"⚠️ Skipping step {step} — loss={loss.item()}")
        optimizer.zero_grad()
        continue


    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_d.parameters(), max_norm=1.0)
    optimizer.step()

    total_loss += loss.item()
    total_dist_loss += distill_loss.item()
    total_sample_loss += sample_loss.item()

    print(
      f"Epoch [{epoch+1}/{EPOCHS}]"
      f"Step [{step+1}/{len(dataloader)}] "
      f"Loss: {loss.item():.4f} | "
      f"Distill Loss: {distill_loss.item():.4f} | "
      f"Sample Loss: {sample_loss.item():.4f} | "
    )

    if (step + 1) % 50 == 0:
      pass


    if (step + 1) % SAVE_EVERY_N == 0:
      checkpoint_dir = f"{CHECKPOINT_DIR}/epoch{epoch+1}_step{step+1}"
      os.makedirs(checkpoint_dir, exist_ok=True)

      model_d.save_pretrained(checkpoint_dir)
      tokenizer_d.save_pretrained(checkpoint_dir)

      torch.save({
          "epoch" : epoch,
          "step" : step,
          "optimizer_state_dict" : optimizer.state_dict(),
          "loss" : loss.item()
      }, f"{checkpoint_dir}/training_state.pt")

      checkpoints = sorted([
          d for d in os.listdir(CHECKPOINT_DIR)
          if os.path.isdir(f"{CHECKPOINT_DIR}/{d}")
      ])

      while len(checkpoints) > CHECKPOINT_WINDOW_LEN:
        old_checkpoint = checkpoints.pop(0)
        old_path = f"{CHECKPOINT_DIR}/{old_checkpoint}"
        shutil.rmtree(old_path)

      print(f"Saved at step {step}")

  # --- Epoch summary ---
  avg_loss      = total_loss / len(dataloader)
  avg_dist_loss = total_dist_loss / len(dataloader)
  avg_sample_loss = total_sample_loss / len(dataloader)

  print(f"\n{'='*60}")
  print(f"Epoch {epoch+1} Complete")
  print(f"  Avg Total Loss : {avg_loss:.4f}")
  print(f"  Avg Distill    : {avg_dist_loss:.4f}")
  print(f"  Avg Task Loss  : {avg_sample_loss:.4f}")
  print(f"{'='*60}\n")

os.makedirs(SAVE_DIR, exist_ok=True)
model_d.save_pretrained(SAVE_DIR)
tokenizer_d.save_pretrained(SAVE_DIR)

torch.save({
    "epoch" : epoch,
    "step" : step,
    "optimizer_state_dict" : optimizer.state_dict(),
    "loss" : loss.item()
}, f"{SAVE_DIR}/training_state.pt")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
torch.Size([2, 512])
log_prob_t NaN: False
log_prob_d NaN: False
log_prob_t min: -18.5176
log_prob_d min: -19.0887
distill_loss: 6.2957
sample_loss:  15.7284
Epoch [1/3]Step [1/10011] Loss: 7.2390 | Distill Loss: 6.2957 | Sample Loss: 15.7284 | 
torch.Size([2, 512])
⚠️ Skipping step 1 — loss=nan
torch.Size([2, 512])
⚠️ Skipping step 2 — loss=nan
torch.Size([2, 512])
⚠️ Skipping step 3 — loss=nan
torch.Size([2, 512])
⚠️ Skipping step 4 — loss=nan
torch.Size([2, 512])
⚠️ Skipping step 5 — loss=nan
torch.Size([2, 512])
⚠️ Skipping step 6 — loss=nan
torch.Size([2, 512])
⚠️ Skipping step 7 — loss=nan
torch.Size([2, 512])
⚠️ Skipping step 8 — loss=nan
torch.Size([2, 512])
⚠️ Skipping step 9 — loss=nan
torch.Size([2, 512])
⚠️ Skipping step 10 — loss=nan
torch.Size([2, 512])
⚠️ Skipping step 11 — loss=nan
torch.Size([2, 512])
⚠️ Skipping step 12 — loss=nan
torch.Size

KeyboardInterrupt: 

In [45]:
from transformers import AutoModelForCausalLM

name_d = "Qwen/Qwen2.5-0.5B-Instruct"

del model_d
torch.cuda.empty_cache()

model_d = AutoModelForCausalLM.from_pretrained(
    name_d,
    torch_dtype=torch.float16,
    device_map="auto"
)

model_d.train()
print("✅ model_d reloaded")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ model_d reloaded
